In [1]:
import os
import glob
import sys

import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

In [2]:
from legnet_embedding import LegNetEmbedding
sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from pl_regressor import RNARegressor

## Loading data

In [3]:
utr_type = "utr5"

In [4]:
if utr_type == "utr5":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr5-deltas-epoch=9-step=840.ckpt"
elif utr_type == "utr3":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"
else:
    raise ValueError

In [5]:
PATH_FROM = f'../../predictor/regression_multiple/{utr_type.upper()}_zscores_replicateagg.csv'
df = pd.read_csv(PATH_FROM)

In [6]:
df.head(10)

,seq,cell_type,fold,1,2,3,4,mass_center,mass_center_mean,diff,zscore,mass_center_std
0,AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCA...,c1,train,52.125480,52.593756,49.272745,47.834227,2.459879,2.368279,0.091601,0.779921,0.117449
1,AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCA...,c17,train,46.974545,51.103420,46.791818,34.483872,2.383516,2.368279,0.015238,0.129738,0.117449
2,AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCA...,c2,train,50.231043,56.194372,60.956634,48.531620,2.499222,2.368279,0.130943,1.114897,0.117449
3,AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCA...,c4,train,72.587265,56.853170,47.180959,36.766505,2.225536,2.368279,-0.142742,-1.215358,0.117449
4,AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCA...,c6,train,63.071485,42.937131,41.591609,35.795388,2.273239,2.368279,-0.095039,-0.809197,0.117449
5,AAAAACCAGCGCTGCCCTCTTTGAAAGCCAGGGAGCATCATTCATT...,c1,train,58.204492,47.247208,42.832102,42.315960,2.363377,2.471239,-0.107862,-1.111272,0.097062
6,AAAAACCAGCGCTGCCCTCTTTGAAAGCCAGGGAGCATCATTCATT...,c17,train,36.961565,39.504869,28.262452,34.638455,2.434663,2.471239,-0.036576,-0.376832,0.097062
7,AAAAACCAGCGCTGCCCTCTTTGAAAGCCAGGGAGCATCATTCATT...,c2,train,35.994504,39.243537,46.870492,47.701708,2.625872,2.471239,0.154633,1.593139,0.097062
8,AAAAACCAGCGCTGCCCTCTTTGAAAGCCAGGGAGCATCATTCATT...,c4,train,57.545992,49.132772,43.444637,52.263239,2.446794,2.471239,-0.024445,-0.251848,0.097062
9,AAAAACCAGCGCTGCCCTCTTTGAAAGCCAGGGAGCATCATTCATT...,c6,train,55.280457,38.733418,48.822489,50.051401,2.485489,2.471239,0.014250,0.146813,0.097062


In [7]:
ct_classes = df["cell_type"].unique()
num_classes = ct_classes.shape[0]
num_classes

5

In [8]:
batch_size = 1024

In [9]:
num_workers = 32

In [10]:
test_set = utrdata.UTRData(
    df=df,
    predict_cols=[],
    features=("sequence", "positional", "conditions"),
    construct_type=utr_type,
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)

In [11]:
# Creating DataLoaders
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

In [12]:
ckpt = torch.load(MODEL_PATH)
ckpt["hyper_parameters"]["model_class"] = LegNetEmbedding

loaded_model = RNARegressor(**ckpt["hyper_parameters"])
loaded_model.load_state_dict(ckpt["state_dict"])

<All keys matched successfully>

In [13]:
progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)
trainer = pl.Trainer(
    callbacks=[progressbar_callback],
    logger=False,
    accelerator="gpu",
    devices=[1],
    deterministic=True,
    # gradient_clip_val=1e-5,
    # gradient_clip_algorithm="norm",
)

prediction = trainer.predict(model=loaded_model, dataloaders=dl_test)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

In [14]:
pred_tuples, _ = zip(*prediction)
embed, pred = zip(*pred_tuples)
embed = torch.concat(embed).numpy()
pred = torch.concat(pred).numpy()

In [15]:
seq_embedding = embed.reshape(-1, num_classes * embed.shape[-1], order="C")
seq_embedding

array([[0.14585872, 0.1690155 , 0.10657468, ..., 0.14074169, 0.19631097,
        0.1188376 ],
       [0.14589424, 0.16809288, 0.10318821, ..., 0.15090004, 0.20081612,
        0.1262806 ],
       [0.11498762, 0.12280534, 0.10574961, ..., 0.08116944, 0.19868694,
        0.09796766],
       ...,
       [0.13197404, 0.15606657, 0.13565698, ..., 0.1529268 , 0.18277545,
        0.12876199],
       [0.14815228, 0.15551583, 0.16194879, ..., 0.15554127, 0.1582575 ,
        0.22158746],
       [0.18753426, 0.14008607, 0.12915966, ..., 0.1456767 , 0.1842768 ,
        0.15711778]], dtype=float32)

In [16]:
pred_mass_center = pred[:, 1].reshape(-1, num_classes)
pred_mass_center

array([[2.4839292, 2.457715 , 2.5757082, 2.5171173, 2.4982133],
       [2.453609 , 2.4326696, 2.5356035, 2.4216824, 2.3940763],
       [2.2592278, 2.1608663, 2.4535325, 2.2847404, 2.1600118],
       ...,
       [2.4540765, 2.4183986, 2.510536 , 2.4198792, 2.3832626],
       [2.6218646, 2.711904 , 2.5904565, 2.615633 , 2.6705673],
       [2.5483668, 2.5778651, 2.5561998, 2.5409958, 2.5396686]],
      dtype=float32)

In [17]:
result = df.pivot(columns="cell_type", index="seq", values="mass_center")
result.columns = pd.MultiIndex.from_product([["mass_center"], result.columns])
result.head()

mass_center            \
cell_type                                                   c1       c17   
seq                                                                        
AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCAGTTT    2.459879  2.383516   
AAAAACCAGCGCTGCCCTCTTTGAAAGCCAGGGAGCATCATTCATTTAGC    2.363377  2.434663   
AAAAACCCAGTCAGCCTGGCTCCTGTTGAATAGTCTACCCCCCTTGCACT    2.530784  2.210211   
AAAAAGAAGCCTCTGAACCCGCGCCGGCCCGCAGCCCCCGTGCCTTCCGG    2.601935  2.602807   
AAAAATCCCTCCCCGGCGGCGGCGGCGGCGGCGGCGGCGCCGGCGGTGGT    2.607835  2.473463   

                                                                        \
cell_type                                                 c2        c4   
seq                                                                      
AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCAGTTT  2.499222  2.225536   
AAAAACCAGCGCTGCCCTCTTTGAAAGCCAGGGAGCATCATTCATTTAGC  2.625872  2.446794   
AAAAACCCAGTCAGCCTGGCTCCTGTTGAATAGTCTACCCCCCTTGCACT  2.496275  2.329457   
AAAAAGAAGCCTCTGAACCCGCGCCGGCCCGCAGCCCCCGTGCCTTCCGG  2.631921  2.498906   
AAAAATCCCTCCCCGGCGGCGGCGGCGGCGGCGGCGGCGCCGGCGGTGGT  2.562370  2.496690   

                                                              
cell_type                                                 c6  
seq                                                           
AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCAGTTT  2.273239  
AAAAACCAGCGCTGCCCTCTTTGAAAGCCAGGGAGCATCATTCATTTAGC  2.485489  
AAAAACCCAGTCAGCCTGGCTCCTGTTGAATAGTCTACCCCCCTTGCACT  2.368066  
AAAAAGAAGCCTCTGAACCCGCGCCGGCCCGCAGCCCCCGTGCCTTCCGG  2.636880  
AAAAATCCCTCCCCGGCGGCGGCGGCGGCGGCGGCGGCGCCGGCGGTGGT  2.660436

In [18]:
result[
    pd.MultiIndex.from_product([["pred_mass_center"], result["mass_center"].columns])
] = pred_mass_center
result["pred_mass_center"].head()

cell_type,c1,c17,c2,c4,c6
seq,,,,,
AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCAGTTT,2.483929,2.457715,2.575708,2.517117,2.498213
AAAAACCAGCGCTGCCCTCTTTGAAAGCCAGGGAGCATCATTCATTTAGC,2.453609,2.432670,2.535604,2.421682,2.394076
AAAAACCCAGTCAGCCTGGCTCCTGTTGAATAGTCTACCCCCCTTGCACT,2.259228,2.160866,2.453532,2.284740,2.160012
AAAAAGAAGCCTCTGAACCCGCGCCGGCCCGCAGCCCCCGTGCCTTCCGG,2.547473,2.582592,2.592539,2.566762,2.527392
AAAAATCCCTCCCCGGCGGCGGCGGCGGCGGCGGCGGCGCCGGCGGTGGT,2.515894,2.464072,2.510534,2.594868,2.500021


In [19]:
seq_embedding_df = pd.DataFrame(
    seq_embedding,
    columns=pd.MultiIndex.from_product([
        ["embedding"],
        [f"{ct}_{i:03d}" for ct in result["mass_center"].columns for i in range(embed.shape[-1])]
    ]),
    index=result.index)
result = pd.concat([result, seq_embedding_df], axis=1)

In [20]:
result["embedding"]

,c1_000,c1_001,c1_002,c1_003,c1_004,c1_005,c1_006,c1_007,c1_008,c1_009,...,c6_022,c6_023,c6_024,c6_025,c6_026,c6_027,c6_028,c6_029,c6_030,c6_031
seq,,,,,,,,,,,,,,,,,,,,,
AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCAGTTT,0.145859,0.169015,0.106575,0.143215,0.178238,0.116094,0.102426,0.157625,0.140179,0.136094,...,0.089209,0.146829,0.131849,0.145223,0.135439,0.074136,0.185570,0.140742,0.196311,0.118838
AAAAACCAGCGCTGCCCTCTTTGAAAGCCAGGGAGCATCATTCATTTAGC,0.145894,0.168093,0.103188,0.142278,0.166155,0.111113,0.104138,0.147460,0.135266,0.124739,...,0.060380,0.119937,0.135773,0.154515,0.138657,0.041728,0.167132,0.150900,0.200816,0.126281
AAAAACCCAGTCAGCCTGGCTCCTGTTGAATAGTCTACCCCCCTTGCACT,0.114988,0.122805,0.105750,0.083006,0.170469,0.109989,0.073920,0.148184,0.113963,0.107330,...,-0.051657,0.082465,0.105992,0.204932,0.112303,-0.037531,0.220111,0.081169,0.198687,0.097968
AAAAAGAAGCCTCTGAACCCGCGCCGGCCCGCAGCCCCCGTGCCTTCCGG,0.135387,0.136024,0.104153,0.153658,0.125807,0.118985,0.138735,0.122927,0.131779,0.128430,...,0.230138,0.121359,0.147563,0.162093,0.123698,0.059676,0.158437,0.140021,0.184280,0.178119
AAAAATCCCTCCCCGGCGGCGGCGGCGGCGGCGGCGGCGCCGGCGGTGGT,0.198645,0.143094,0.122580,0.180100,0.153296,0.145580,0.166727,0.146817,0.123465,0.136053,...,0.163745,0.131338,0.149150,0.165218,0.166815,-0.002331,0.160150,0.196978,0.150303,0.160402
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTGGCTCCGCGGGCTCGCCCTCCGCTCCCTCTGCCAGCGGCGCCAGAC,0.167701,0.145641,0.169393,0.146522,0.139841,0.195750,0.177767,0.107951,0.138944,0.149204,...,0.327981,0.115276,0.146174,0.092024,0.107935,0.111707,0.131504,0.141087,0.137748,0.210550
TTTTGTGCCGAGGCACCTACACACCTCCCGTCCTCTCTGCCAGATCGCGG,0.151508,0.138200,0.143521,0.132239,0.146949,0.152162,0.144359,0.121634,0.120039,0.122528,...,0.235510,0.109172,0.125432,0.136193,0.124520,0.092169,0.142957,0.137543,0.164568,0.175212
TTTTTAAAACACAGAGCAAGCGCCAGAGGCGTCGGCATCCCAGGTGTCGC,0.131974,0.156067,0.135657,0.116303,0.168999,0.148175,0.115408,0.135505,0.124792,0.135443,...,0.092826,0.094141,0.126067,0.164168,0.111655,0.064347,0.155565,0.152927,0.182775,0.128762


### Counting k-mers

In [21]:
sys.path.append("../../benchmark/kmers")
from kmer_model import KmerLinearRegressor

In [22]:
kmer_dfs = []
for k in [1, 2, 3]:
    kmerreg = KmerLinearRegressor(kmer_length=k)
    kmer_df_k = kmerreg.count_kmers(result.index.values)
    kmer_df_k.columns = pd.MultiIndex.from_product([
        ["kmer_counts"],
        kmer_df_k.columns
    ])
    kmer_dfs.append(kmer_df_k)
    print(f"k={k} done")
result = pd.concat([result] + kmer_dfs, axis=1)

k=1 done
k=2 done
k=3 done


In [23]:
result["kmer_counts"]

,A,C,G,T,AA,AC,AG,AT,CA,CC,...,TCG,TCT,TGA,TGC,TGG,TGT,TTA,TTC,TTG,TTT
seq,,,,,,,,,,,,,,,,,,,,,
AAAAACAACCAGAGGCTGCTCTGCTTGAGGGTGAAGCCGCCTCCCAGTTT,13,14,13,10,6,2,5,0,3,5,...,0,1,2,2,0,0,0,0,1,1
AAAAACCAGCGCTGCCCTCTTTGAAAGCCAGGGAGCATCATTCATTTAGC,15,14,10,11,6,1,5,3,5,4,...,0,1,1,1,0,0,1,1,1,2
AAAAACCCAGTCAGCCTGGCTCCTGTTGAATAGTCTACCCCCCTTGCACT,12,18,8,12,5,3,3,1,3,9,...,0,1,1,1,1,1,0,0,2,0
AAAAAGAAGCCTCTGAACCCGCGCCGGCCCGCAGCCCCCGTGCCTTCCGG,10,22,13,5,6,1,3,0,1,12,...,0,1,1,1,0,0,0,1,0,0
AAAAATCCCTCCCCGGCGGCGGCGGCGGCGGCGGCGGCGCCGGCGGTGGT,5,18,23,4,4,0,0,1,0,6,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTGGCTCCGCGGGCTCGCCCTCCGCTCCCTCTGCCAGCGGCGCCAGAC,3,23,14,10,0,1,2,0,2,8,...,1,1,0,1,1,0,0,0,1,2
TTTTGTGCCGAGGCACCTACACACCTCCCGTCCTCTCTGCCAGATCGCGG,7,20,11,12,0,4,2,1,4,7,...,1,2,0,2,0,1,0,0,1,2
TTTTTAAAACACAGAGCAAGCGCCAGAGGCGTCGGCATCCCAGGTGTCGC,13,14,14,9,4,2,6,1,6,3,...,2,0,0,0,0,1,1,0,0,3


In [24]:
result.to_parquet(f"embeddings_mapperless_{utr_type}.parquet")